# Logistic Regression
It is the same as linear regression in formula (with a minor difference), but it outputs a classification based on the percentage it computes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from meta import *

In [2]:
print("Get the previously wrangled dataset")
print("-"*20)
df = pd.read_csv(savepath + r"/student_eda.csv")

Get the previously wrangled dataset
--------------------


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6378 entries, 0 to 6377
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6378 non-null   int64
 1   Attendance                  6378 non-null   int64
 2   Parental_Involvement        6378 non-null   int64
 3   Access_to_Resources         6378 non-null   int64
 4   Extracurricular_Activities  6378 non-null   int64
 5   Sleep_Hours                 6378 non-null   int64
 6   Previous_Scores             6378 non-null   int64
 7   Motivation_Level            6378 non-null   int64
 8   Internet_Access             6378 non-null   int64
 9   Tutoring_Sessions           6378 non-null   int64
 10  Family_Income               6378 non-null   int64
 11  Teacher_Quality             6378 non-null   int64
 12  School_Type                 6378 non-null   int64
 13  Peer_Influence              6378 non-null   int64
 14  Physical

In [4]:
print("Threshold is 70, meaning if a student scores 70 or above, they pass. Otherwise they fail")
print("-"*20)
df['Pass_Fail'] = (df['Exam_Score'] >= 70).astype(int)

print("Pass/Fail Dağılımı:")
print(df['Pass_Fail'].value_counts())
print("\nOranlar:")
print(df['Pass_Fail'].value_counts(normalize=True) * 100)

Threshold is 70, meaning if a student scores 70 or above, they pass. Otherwise they fail
--------------------
Pass/Fail Dağılımı:
Pass_Fail
0    4797
1    1581
Name: count, dtype: int64

Oranlar:
Pass_Fail
0    75.211665
1    24.788335
Name: proportion, dtype: float64


In [5]:
X = df.drop(['Exam_Score', 'Pass_Fail'], axis=1)
y = df['Pass_Fail']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nÖzelliklerin listesi:\n{X.columns.tolist()}")

X shape: (6378, 19)
y shape: (6378,)

Özelliklerin listesi:
['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")
print(f"\nEğitim setinde Pass/Fail oranı:\n{y_train.value_counts()}")
print(f"\nTest setinde Pass/Fail oranı:\n{y_test.value_counts()}")

Eğitim seti boyutu: (5102, 19)
Test seti boyutu: (1276, 19)

Eğitim setinde Pass/Fail oranı:
Pass_Fail
0    3837
1    1265
Name: count, dtype: int64

Test setinde Pass/Fail oranı:
Pass_Fail
0    960
1    316
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled ortalaması: {X_train_scaled.mean():.6f}")
print(f"X_train_scaled standart sapması: {X_train_scaled.std():.6f}")

X_train_scaled ortalaması: 0.000000
X_train_scaled standart sapması: 1.000000


In [8]:
from sklearn.linear_model import LogisticRegression
import numpy as np

print("📋 Veri Kontrol:")
print(f"  X_train shape: {X_train_scaled.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  X_train NaN sayısı: {np.isnan(X_train_scaled).sum()}")
print(f"  y_train unique: {np.unique(y_train)}")

try:
    model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
    
    model.fit(X_train_scaled, y_train)
    
    print("\nLogistic Regression modeli başarıyla eğitildi!")
    print(f"   Model parametreleri: {model.coef_.shape[0]} özellik")
    
except Exception as e:
    print(f"\nModel eğitim hatası: {str(e)}")
    import traceback
    traceback.print_exc()
    raise

📋 Veri Kontrol:
  X_train shape: (5102, 19)
  y_train shape: (5102,)
  X_train NaN sayısı: 0
  y_train unique: [0 1]

Logistic Regression modeli başarıyla eğitildi!
   Model parametreleri: 1 özellik


In [9]:
y_pred = model.predict(X_test_scaled)

y_pred_proba = model.predict_proba(X_test_scaled)

print("İlk 10 tahmin:")
print(f"Tahmin edilen sınıflar: {y_pred[:10]}")
print(f"Geçme olasılıkları: {y_pred_proba[:10, 1]}")

İlk 10 tahmin:
Tahmin edilen sınıflar: [0 1 1 0 0 0 0 0 0 1]
Geçme olasılıkları: [4.06491947e-03 9.95904173e-01 9.45953031e-01 5.09283678e-06
 4.90770129e-05 2.06386729e-04 4.21121717e-05 1.36706393e-03
 5.89733676e-06 8.21241785e-01]


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 50)
print("MODEL PERFORMANS METRİKLERİ")
print("=" * 50)
print("Accuracy (Correct Predictions / Total Predictions)")
print(f"{accuracy:.4f} ({accuracy*100:.2f}%)")
print("Precision (Correct Positive Predictions / All Positive Predictions)")
print(f"{precision:.4f} ({precision*100:.2f}%)")
print("Recall (Correct Positive Predictions / All Actual Positive Instances)")
print(f"{recall:.4f} ({recall*100:.2f}%)")
print("F1 (Harmonic Mean of Precision and Recall)")
print(f"{f1:.4f}")
print("\n" + "=" * 50)
print("KARIŞIKLIK MATRİSİ (Confusion Matrix)")
print("=" * 50)
cm = confusion_matrix(y_test, y_pred)
print(f"Doğru Negatif (TN):  {cm[0, 0]}  | Yanlış Pozitif (FP): {cm[0, 1]}")
print(f"Yanlış Negatif (FN): {cm[1, 0]}  | Doğru Pozitif (TP):  {cm[1, 1]}")

MODEL PERFORMANS METRİKLERİ
Accuracy (Correct Predictions / Total Predictions)
0.9835 (98.35%)
Precision (Correct Positive Predictions / All Positive Predictions)
0.9743 (97.43%)
Recall (Correct Positive Predictions / All Actual Positive Instances)
0.9589 (95.89%)
F1 (Harmonic Mean of Precision and Recall)
0.9665

KARIŞIKLIK MATRİSİ (Confusion Matrix)
Doğru Negatif (TN):  952  | Yanlış Pozitif (FP): 8
Yanlış Negatif (FN): 13  | Doğru Pozitif (TP):  303


In [11]:
X_all_scaled = scaler.transform(X)
print("X scaled.")
print("-"*20)
y_pred_all = model.predict(X_all_scaled)
print("Predictions done.")
print("-"*20)

df['Predicted_Pass_Fail'] = y_pred_all

y_pred_proba_all = model.predict_proba(X_all_scaled)
df['Pass_Probability'] = y_pred_proba_all[:, 1]

print("=" * 60)
print("RESULTS")
print("=" * 60)

pass_count = (y_pred_all == 1).sum()
fail_count = (y_pred_all == 0).sum()
total_count = len(y_pred_all)

pass_percent = (pass_count / total_count) * 100
fail_percent = (fail_count / total_count) * 100

print(f"Toplam Öğrenci: {total_count}")
print(f"Geçen:      {pass_count:3d} ({pass_percent:6.2f}%)")
print(f"Kalan:      {fail_count:3d} ({fail_percent:6.2f}%)")
print("\n" + "=" * 60)

X scaled.
--------------------
Predictions done.
--------------------
RESULTS
Toplam Öğrenci: 6378
Geçen:      1546 ( 24.24%)
Kalan:      4832 ( 75.76%)



In [12]:
import pandas as pd

coefficients_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

print("\n" + "=" * 60)
print("MODELİN ÖZELLİK KATSAYILARı")
print("=" * 60)
print("(Pozitif = Geçme eğilimi, Negatif = Kalma eğilimi)")
print("=" * 60)
print(coefficients_df.to_string(index=False))
print("=" * 60)


MODELİN ÖZELLİK KATSAYILARı
(Pozitif = Geçme eğilimi, Negatif = Kalma eğilimi)
                   Feature  Coefficient
                Attendance     4.768174
             Hours_Studied     3.746252
       Access_to_Resources     1.516052
           Previous_Scores     1.428094
      Parental_Involvement     1.359342
         Tutoring_Sessions     1.262661
            Peer_Influence     0.846248
  Parental_Education_Level     0.798704
          Motivation_Level     0.797607
             Family_Income     0.735994
        Distance_from_Home     0.706856
           Teacher_Quality     0.700917
           Internet_Access     0.581498
     Learning_Disabilities    -0.524108
Extracurricular_Activities     0.518298
         Physical_Activity     0.436415
                    Gender    -0.060045
               School_Type    -0.053324
               Sleep_Hours     0.022039


# Interactive Cell
Start this cell and enter student data to see if they fail or pass.

In [13]:
required_vars = ["model", "scaler", "X"]
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    print("Eksik değişkenler:", missing_vars)
    print("Önce veri yükleme, model eğitme ve scaler hücrelerini çalıştırın.")
else:
    def get_valid_input(prompt, min_val, max_val):
        while True:
            user_input = input(prompt)
            try:
                value = float(user_input)
            except ValueError:
                print("Lütfen sayısal bir değer girin.")
                continue
            if not (min_val <= value <= max_val):
                print(f"Değer {min_val} ile {max_val} arasında olmalı.")
                continue
            return value

    print("Öğrenci başarı tahmini")
    print("Aşağıdaki değerleri sırayla girin.\n")

    input_values = {}
    for col in X.columns:
        if col == "Gender":
            value = get_valid_input(f"{col} (0=Kadin, 1=Erkek): ", 0, 1)
        elif col == "School_Type":
            value = get_valid_input(f"{col} (0=Devlet, 1=Ozel): ", 0, 1)
        elif col in ["Learning_Disabilities", "Internet_Access", "Extracurricular_Activities"]:
            value = get_valid_input(f"{col} (0=Hayir, 1=Evet): ", 0, 1)
        elif col in ["Peer_Influence", "Parental_Education_Level", "Distance_from_Home"]:
            value = get_valid_input(f"{col} (0-2 arasinda): ", 0, 2)
        else:
            value = get_valid_input(f"{col} (0-2 arasinda): ", 0, 2)
        input_values[col] = value

    input_df = pd.DataFrame([input_values])[X.columns]
    input_scaled = scaler.transform(input_df)

    prediction = model.predict(input_scaled)[0]
    proba = model.predict_proba(input_scaled)[0]

    print("\nSonuc")
    print("-----")
    if prediction == 1:
        print("Tahmin: Gecti")
    else:
        print("Tahmin: Kaldi")
    print(f"Gecme olasiligi: {proba[1]*100:.2f}%")
    print(f"Kalma olasiligi: {proba[0]*100:.2f}%")

Öğrenci başarı tahmini
Aşağıdaki değerleri sırayla girin.


Sonuc
-----
Tahmin: Kaldi
Gecme olasiligi: 0.00%
Kalma olasiligi: 100.00%
